### ~

In [90]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#
from pathlib import Path
import shutil
import subprocess
import builtins



ROOT = Path.cwd()

def here():
    return ROOT

def make_dir(path):
    p = ROOT / path
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_file(path, content=""):
    p = ROOT / path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return p

def remove_path(path):
    p = ROOT / path
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()
    return p

def run_cmd(*cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode
#
import os
from pathlib import Path
import shutil
import subprocess


class WorkspaceBase:
    @staticmethod
    def get_notebook_name():
        notebook_files = [f for f in os.listdir('.') if f.endswith('.ipynb')]
        if notebook_files:
            thienotebooktitleworkspace = os.path.splitext(notebook_files[0])[0]
        else:
            thienotebooktitleworkspace = "Untitled Notebook"
        return thienotebooktitleworkspace

    def get_root(self):
        return self.root

    def make_folder(self, name):
        folder = self.root / name
        folder.mkdir(parents=True, exist_ok=True)
        return folder

    def write_text(self, path, text):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        return path

    def read_text(self, path):
        return Path(path).read_text(encoding="utf-8")

    def run_cmd(self, *args):
        result = subprocess.run(args, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)

    def __init__(self, root=None):
        if root is None:
            root = f"{self.get_notebook_name()}_workspace"
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

class Problem_(WorkspaceBase):
    def __init__(self, name, files=None, root=None):
        super().__init__(root=root)
        self.name = name
        self.files = files or {}
        self.folder = self.make_folder(self.safe_name())

    def safe_name(self):
        return self.name.strip().lower().replace(" ", "_")

    def add_file(self, relative_path, content):
        if content and content[0] == '\n':
            content = content[1:]
        self.files[str(relative_path)] = content

    def get_file(self, relative_path):
        return self.files.get(str(relative_path))

    def remove_file(self, relative_path):
        self.files.pop(str(relative_path), None)

    def save(self, clear_first=False):
        if clear_first:
            self.reset_folder(self.folder)
        for relative_path, content in self.files.items():
            full_path = self.folder / relative_path
            self.write_text(full_path, content)

    def run(self, command, cwd=None, inputs=None):
        """Run a shell command. Pass inputs as a list of strings for stdin."""
        if cwd is None:
            cwd = self.folder
        stdin_text = "\n".join(str(v) for v in inputs) + "\n" if inputs else None
        result = subprocess.run(command, cwd=cwd, capture_output=True, text=True,
                                shell=True, input=stdin_text)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)

    def reset_folder(self, folder):
        folder = Path(folder)
        if folder.exists():
            shutil.rmtree(folder)
        folder.mkdir(parents=True, exist_ok=True)
        return folder

    def load(self):
        loaded = {}
        for relative_path in self.list_all_files(self.folder):
            full_path = self.folder / relative_path
            loaded[relative_path] = self.read_text(full_path)
        self.files = loaded
        return loaded

    def summary(self):
        return {
            "name": self.name,
            "folder": str(self.folder),
            "file_count": len(self.files),
            "files": sorted(self.files.keys())
        }

    def __repr__(self):
        return f"Problem(name={self.name!r}, files={len(self.files)}, folder={str(self.folder)!r})"

class Problem(Problem_):
    def __init__(self, name, files=None, root=None):
        super().__init__(name, files, root)
        self.reset_folder(self.folder)

    def runpy(self, command, cwd=None, inputs=None):
        """Run a python file. Pass inputs as a list of strings for stdin.
        Example: _.runpy("main.py", inputs=["8", "3"])
        """
        print(f"Ran command: python {command} in {cwd or self.folder}")
        self.run(f"python {command}", cwd, inputs=inputs)

    def create(self, relative_path, content):
        self.add_file(relative_path, content)
        self.save()
        print(f"Created file: {relative_path} with content:\n{content}")

    def create2(self, relative_path, content, inputs=None):
        self.create(relative_path, content)
        self.runpy(relative_path, cwd=self.folder, inputs=inputs)

    def answer(self, content):
        self.add_file("ANSWER.md", content)
        self.save()
        print(f"Saved answer to ANSWER.md:\n{content}")
#
def setin(*inputs):
    """
    A helper function to set test inputs for the input() function.
    Usage:
    setin("input1", "input2", "input3")
    This will set up the input() function to return "input1" on the first call,
    "input2" on the second call, and so on.
    To reset to normal input behavior, call setin() with no arguments or None:
    setin()
    """
    if inputs:
        input_iter = iter(inputs)
        def mock_input(prompt=""):
            try:
                value = next(input_iter)
                print(f"{prompt}{value}")
                return value
            except StopIteration:
                raise EOFError("No more inputs for testing")
        builtins.input = mock_input
    else:
        builtins.input = builtins._original_input_backup
              
#

### ~

In [76]:
'''11.1 Modules
Module basics
The interactive Python interpreter provides the most basic way to execute Python code. However, all of the defined variables, functions, classes, etc., are lost when a programmer closes the interpreter. Thus, a programmer will typically write Python code in a file, and then pass that file as input to the interpreter. Such a file is called a script.

A programmer may find themselves writing the same function over and over again in multiple scripts, or creating very long and difficult-to-maintain scripts. A solution is to use a module, which is a file containing Python code that can be imported and used by scripts, other modules, or the interactive interpreter. To import a module means to execute the code contained by the module and make the definitions within that module available for use by the importing program.

participation activity
11.1.1: A module is a file containing Python statements and definitions that can be used by other Python sources.


1

2

The functions can instead be defined in another file. The file can be imported as a "module".
def fct():
   # ...
def sq():   
   # ...
x = fct() * sq()
# ...
script1.py
def fct():
   # ...
def sq():   
   # ...
x = fct() / sq()
# ...
script2.pyfuncs.py
def fct():
   # ...
def sq():   
   # ...
script1.py
import funcs
script2.py
import funcs
x = funcs.fct() * 
      funcs.sq()
x = funcs.fct() / 
      funcs.sq()
fct()sq()
*
fct()sq()
/
def fct():
   # ...
def sq():   
   # ...
Static figure: Begin Python code (script1.py, before using a module): def fct(): # ... def sq(): # ... x = fct() * sq() # ... End Python code. Begin Python code (script2.py, before using a module): def fct(): # ... def sq(): # ... x = fct() / sq() # ... End Python code. Begin Python code (funcs.py module): def fct(): # ... def sq(): # ... End Python code. Begin Python code (script1.py, after using a module): import funcs x = funcs.fct() * funcs.sq() End Python code. Begin Python code (script2.py, after using a module): import funcs x = funcs.fct() / funcs.sq() End Python code. Step 1: A programmer writes scripts containing functions and code using those functions. Multiple scripts might define the same functions. script1.py and script2.py are shown defining the same functions fct() and sq(). Step 2: The functions can instead be defined in another file. The file can be imported as a "module". The module funcs.py is created with the same function definitions in both script1.py and script2.py. script1.py is rewritten to instead import the functions from the funcs.py module. script1.py is executed, starting with the import statement. To execute the calculation for x, the function names from the module are shown above the function names in script1.py. The process repeats for script2.py using the same module functions, but with a different calculation for x.

Captions
A programmer writes scripts containing functions and code using those functions. Multiple scripts might define the same functions.
The functions can instead be defined in another file. The file can be imported as a "module".
Playing step 2: The functions can instead be defined in another file. The file can be imported as a "module". Step finished playing

Feedback?

A module's filename should end with ".py"; otherwise, the interpreter will not be able to import the module. The module_name item should match the filename of the module, but without the .py extension. Ex: If a programmer wants to import a module whose filename is HTTPServer.py, the import statement import HTTPServer would be used. Note that import statements are case-sensitive; thus, import ABC is distinct from import aBc.

The interpreter must also be able to find the module to import. The simplest solution is to keep modules in the same directory as the executing script; however, the interpreter can also search the computer's file system for the modules. Later material covers these search mechanisms.

Good practice is to place import statements at the top of a file. There are few useful instances of placing import statements in any location other than the top. The benefit of placing import statements at the top is that a reader of the program can quickly identify the modules required for the program to run. A module being required by another program is often called a dependency.'''

_= Problem("participation activity 11.1.1")
_.create("script1.py", '''import funcs
x = funcs.fct() * funcs.sq()''')
_.create("funcs.py", '''def fct():
    return 42
def sq():
    return 7''')
_.runpy("script1.py", cwd=_.folder)


'11.1 Modules\nModule basics\nThe interactive Python interpreter provides the most basic way to execute Python code. However, all of the defined variables, functions, classes, etc., are lost when a programmer closes the interpreter. Thus, a programmer will typically write Python code in a file, and then pass that file as input to the interpreter. Such a file is called a script.\n\nA programmer may find themselves writing the same function over and over again in multiple scripts, or creating very long and difficult-to-maintain scripts. A solution is to use a module, which is a file containing Python code that can be imported and used by scripts, other modules, or the interactive interpreter. To import a module means to execute the code contained by the module and make the definitions within that module available for use by the importing program.\n\nparticipation activity\n11.1.1: A module is a file containing Python statements and definitions that can be used by other Python sources.\n\

Created file: script1.py with content:
import funcs
x = funcs.fct() * funcs.sq()
Created file: funcs.py with content:
def fct():
    return 42
def sq():
    return 7
Ran command: python script1.py in driver_workspace/participation_activity_11.1.1



In [77]:
'''participation activity
11.1.2: Basic importing of modules.
1)
A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
>>>

Check

Show answer
2)
A file containing Python code that is passed as input to the interpreter is called a _______.

Check

Show answer
3)
A _______ is a file containing Python code that can be imported by a script.

Check

Show answer

Feedback?'''

_ = Problem("participation activity 11.1.2")
_.create("tools.py", '''def greet(name):
    return f"Hello, {name}!"''')
_.runpy("tools.py", cwd=_.folder)
_.create("script.py", '''import tools
print(tools.greet("Alice"))''')
_.runpy("script.py", cwd=_.folder)
_.create("ANSWER.md", '''# Participation Activity 11.1.2
          1) A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
            >>> import tools
            2) A file containing Python code that is passed as input to the interpreter is called a _______.
            Answer: script
            3) A _______ is a file containing Python code that can be imported by a script.
            Answer: module''')

'participation activity\n11.1.2: Basic importing of modules.\n1)\nA programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.\n>>>\n\nCheck\n\nShow answer\n2)\nA file containing Python code that is passed as input to the interpreter is called a _______.\n\nCheck\n\nShow answer\n3)\nA _______ is a file containing Python code that can be imported by a script.\n\nCheck\n\nShow answer\n\nFeedback?'

Created file: tools.py with content:
def greet(name):
    return f"Hello, {name}!"
Ran command: python tools.py in driver_workspace/participation_activity_11.1.2

Created file: script.py with content:
import tools
print(tools.greet("Alice"))
Ran command: python script.py in driver_workspace/participation_activity_11.1.2
Hello, Alice!

Created file: ANSWER.md with content:
# Participation Activity 11.1.2
          1) A programmer using the interactive interpreter wants to use a function defined in the file tools.py. Write a statement that imports the content of tools.py into the interpreter.
            >>> import tools
            2) A file containing Python code that is passed as input to the interpreter is called a _______.
            Answer: script
            3) A _______ is a file containing Python code that can be imported by a script.
            Answer: module


In [78]:
'Importing a module\nEvaluating an import statement initiates the following process to load the module:\n\nA check is conducted to determine whether the module has already been imported. If already imported, then the loaded module is used.\nIf not already imported, a new module object is created and inserted in sys.modules.\nThe code in the module is executed in the new module object\'s namespace.\nWhen importing a module, the interpreter first checks to see if that module has already been imported. A dictionary of the loaded modules is stored in sys.modules (available from the sys standard library module). If the module has not yet been loaded, then a new module object is created. A module object is simply a namespace that contains definitions from the module. If the module has already been loaded, then the existing module object is used.\n\nIf a module is not found in sys.modules, then the module is added and the statements within the module\'s code are executed. Definitions in the module\'s code, such as variable assignments and function definitions, are placed in the module\'s namespace. The module is then added to the importing script or module\'s namespace, so the importer can access the definitions. The below animation illustrates.\n\nparticipation activity\n11.1.3: Importing a module.\n\n\n1\n\n2\n\n3\n\n4\n\n5\n\nwebpage has already been imported. Existing module is loaded.\nimport HTTPServer\nimport webpage\n\nmy_ip = HTTPServer.address\n\nwebpage.disp(my_ip)\n\n# ...\nsys.modules\nHTTPServer.py\nwebpage.py\nempty\nHTTPServer\nHTTPServer\nimport webpage\n\naddress = " "\n\n# ...\ndef disp(ip):\n   # ...\n\n# ...\nwebpage\nwebpage\ndisp\nwebpage\naddress\nwebpage\nStatic figure: Begin Python code: import HTTPServer import webpage my_ip = HTTPServer.address webpage.disp(my_ip) # ... End Python code. Begin Python code (HTTPServer.py): import webpage address = " " # ... End Python code. Begin Python code (webpage.py): def disp(ip): # ... # ... End Python code. The sys.modules namespace contains HTTPServer and webpage. The HTTPServer and webpage namespaces are shown with a line drawn to their respective modules in sys.modules. The HTTPServer namespace contains webpage and address. The webpage namespace contains disp. Step 1: sys.modules checks for HTTPServer. A new module object is created. The module is then inserted into sys.modules. sys.modules is initially empty. The main Python code executes the line "import HTTPServer". sys.modules is checked for HTTPServer, a new module is created, and HTTPServer is added to sys.modules. The HTTPServer namespace is initially empty with a line drawn to HTTPServer in sys.modules. Step 2: HTTPServer\'s code is executed in module namespace. sys.modules checks for webpage. The new module object is created and inserted in sys.modules. The HTTPServer code executes the line "import webpage". sys.modules is checked for containing webpage, a new module is created, and webpage is added to sys.modules. The webpage namespace is initially empty with a line drawn to webpage in sys.modules. Step 3: webpage\'s code is executed in module namespace. webpage is added to HTTPServer namespace. webpage\'s code executes, and disp is defined and added to webpage\'s namespace. When webpage\'s code finishes executing, webpage is added to HTTPServer\'s namespace. Step 4: HTTPServer\'s code continues executing. address is defined and added to HTTPServer\'s namespace. Step 5: webpage has already been imported. Existing module is loaded. The main Python code continues and executes the next line "import webpage". sys.modules is checked for webpage, and the existing module is loaded.\n\nCaptions\nsys.modules checks for HTTPServer. A new module object is created. The module is then inserted into sys.modules.\nHTTPServer\'s code is executed in module namespace. sys.modules checks for webpage. The new module object is created and inserted in sys.modules.\nwebpage\'s code is executed in module namespace. webpage is added to HTTPServer namespace.\nHTTPServer\'s code continues executing.\nwebpage has already been imported. Existing module is loaded.\nPlaying step 5: webpage has already been imported. Existing module is loaded. Step finished playing\n\nFeedback?\nExecuting import HTTPServer causes a new module object to be created and added to sys.modules. The code of HTTPServer is executed, which contains another import statement import webpage. Since webpage has not yet been imported, a second new module object is created and added to sys.modules. Execution of the webpage code occurs, which defines a function within the webpage module\'s namespace. Once the webpage module is successfully imported, the execution of HTTPServer\'s code continues, creating new definitions in the HTTPServer module\'s namespace. If the script attempts to import webpage, the already created module object is used.\n\nparticipation activity\n11.1.4: The importing process.\nOrder the events as they occur when the statement import HTTPServer executes, assuming HTTPServer has not been previously imported.\n\nHow to use this tool\nHTTPServer added to importer\'s namespace.\nHTTPServer added to sys.modules.\nModule object created.\nsys.modules checked for HTTPServer.\nHTTPServer code executed.\n1st event\n2nd event\n3rd event\n4th event\n5th event\n\nReset\n\nFeedback?\n'
_ = Problem("participation activity 11.1.4")
_.create("HTTPServer.py", '''import webpage
address = " "# ...''')
_.create("webpage.py", '''def disp(ip):
    print(f"Displaying webpage for IP: {ip}")
# ...''')
_.runpy("HTTPServer.py", cwd=_.folder)
_.create("script.py", '''import HTTPServer
import webpage
my_ip = HTTPServer.address
webpage.disp(my_ip)
# ...''')
_.runpy("script.py", cwd=_.folder)
_.answer('''
         # Participation Activity 11.1.4
         1) sys.modules checked for HTTPServer.
         2) Module object created.
         3) HTTPServer added to sys.modules.
         4) HTTPServer code executed.
         5) HTTPServer added to importer\'s namespace.
         ''')

'Importing a module\nEvaluating an import statement initiates the following process to load the module:\n\nA check is conducted to determine whether the module has already been imported. If already imported, then the loaded module is used.\nIf not already imported, a new module object is created and inserted in sys.modules.\nThe code in the module is executed in the new module object\'s namespace.\nWhen importing a module, the interpreter first checks to see if that module has already been imported. A dictionary of the loaded modules is stored in sys.modules (available from the sys standard library module). If the module has not yet been loaded, then a new module object is created. A module object is simply a namespace that contains definitions from the module. If the module has already been loaded, then the existing module object is used.\n\nIf a module is not found in sys.modules, then the module is added and the statements within the module\'s code are executed. Definitions in the m

Created file: HTTPServer.py with content:
import webpage
address = " "# ...
Created file: webpage.py with content:
def disp(ip):
    print(f"Displaying webpage for IP: {ip}")
# ...
Ran command: python HTTPServer.py in driver_workspace/participation_activity_11.1.4

Created file: script.py with content:
import HTTPServer
import webpage
my_ip = HTTPServer.address
webpage.disp(my_ip)
# ...
Ran command: python script.py in driver_workspace/participation_activity_11.1.4
Displaying webpage for IP:  

Saved answer to ANSWER.md:

         # Participation Activity 11.1.4
         1) sys.modules checked for HTTPServer.
         2) Module object created.
         3) HTTPServer added to sys.modules.
         4) HTTPServer code executed.
         5) HTTPServer added to importer's namespace.
         


In [79]:
'''Using an imported module
Once a module has been imported, the program can access the definitions of a module using attribute reference operations. Ex: my_ip = HTTPServer.address sets my_ip to address defined in HTTPServer.py. The definitions can also be overwritten. Ex: HTTPServer.address = "www.yahoo.com" binds address in HTTPServer to "www.yahoo.com". Note that such changes are temporary and restricted to the current executing Python instance. Ending the program and then re-importing the module would reload the original value of HTTPServer.address.

Consider a file, my_funcs.py, that contains the following:

Figure 11.1.1: Contents of my_funcs.py.
def factorial(num):
    """Calculates and returns the factorial (num!)"""
    x = 1
    for i in range(1, num + 1):
        x *= i

    return x

Feedback?
A programmer can then import my_funcs and use the factorial function as shown below:

Figure 11.1.2: Using factorial from my_funcs.py.
import my_funcs

n = int(input("Enter number: "))
fact = my_funcs.factorial(n)

for i in range(1, n + 1):
    print(i, end=" ")
    if i != n:
        print("*", end=" ")

print(f"= {fact}")
Enter number: 5
1 * 2 * 3 * 4 * 5 = 120
...
Enter number: 3
1 * 2 * 3 = 6

Feedback?
participation activity
11.1.5: Basic usage of imported modules.
Consider a file shapes.py with the following contents:

cr = "#"

def draw_square(size):
    for h in range(size):
        for w in range(size):
            print(cr, end="")
        print() 

def draw_rect(height, width):
    for h in range(height):
        for w in range(width):
            print(cr, end="")
        print()
1)
Complete the import statement to import shapes.py.
import 


Check

Show answer
2)
Complete the statement to call the draw_square function from the shapes module, passing an argument of 3.
shapes.


Check

Show answer
3)
Write a statement that changes the output to use "$" when drawing shapes. (Change the value of shapes.cr.)

Check

Show answer

Feedback?'''
_ = Problem("participation activity 11.1.5")
_.create("shapes.py", 
'''
cr = "#"
def draw_square(size):
    for h in range(size):
        for w in range(size):
            print(cr, end="")
        print()
def draw_rect(height, width):
    for h in range(height):
        for w in range(width):
            print(cr, end="")
        print()''')
_.create("script.py", '''
import shapes
shapes.draw_square(3)
shapes.cr = "$"
shapes.draw_rect(2, 5)''')
_.runpy("script.py", cwd=_.folder)  
_.answer('''
# Participation Activity 11.1.5
1) Complete the import statement to import shapes.py.
import shapes
2) Complete the statement to call the draw_square function from the shapes module, passing an argument of 3.
shapes.draw_square(3)
3) Write a statement that changes the output to use "$" when drawing shapes. (Change the value of shapes.cr.)
shapes.cr = "$"
''')

'Using an imported module\nOnce a module has been imported, the program can access the definitions of a module using attribute reference operations. Ex: my_ip = HTTPServer.address sets my_ip to address defined in HTTPServer.py. The definitions can also be overwritten. Ex: HTTPServer.address = "www.yahoo.com" binds address in HTTPServer to "www.yahoo.com". Note that such changes are temporary and restricted to the current executing Python instance. Ending the program and then re-importing the module would reload the original value of HTTPServer.address.\n\nConsider a file, my_funcs.py, that contains the following:\n\nFigure 11.1.1: Contents of my_funcs.py.\ndef factorial(num):\n    """Calculates and returns the factorial (num!)"""\n    x = 1\n    for i in range(1, num + 1):\n        x *= i\n\n    return x\n\nFeedback?\nA programmer can then import my_funcs and use the factorial function as shown below:\n\nFigure 11.1.2: Using factorial from my_funcs.py.\nimport my_funcs\n\nn = int(input

Created file: shapes.py with content:

cr = "#"
def draw_square(size):
    for h in range(size):
        for w in range(size):
            print(cr, end="")
        print()
def draw_rect(height, width):
    for h in range(height):
        for w in range(width):
            print(cr, end="")
        print()
Created file: script.py with content:

import shapes
shapes.draw_square(3)
shapes.cr = "$"
shapes.draw_rect(2, 5)
Ran command: python script.py in driver_workspace/participation_activity_11.1.5
###
###
###
$$$$$
$$$$$

Saved answer to ANSWER.md:

# Participation Activity 11.1.5
1) Complete the import statement to import shapes.py.
import shapes
2) Complete the statement to call the draw_square function from the shapes module, passing an argument of 3.
shapes.draw_square(3)
3) Write a statement that changes the output to use "$" when drawing shapes. (Change the value of shapes.cr.)
shapes.cr = "$"



In [80]:
'''challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
arithmetic.py
import arithmetic

def calculate(number):
    return number - 5

print(arithmetic.calculate(3))
print(calculate(3))
1
2
3
4
Check
Next
1
2
3
4

Feedback?

challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
arithmetic.py
def calculate(number):
    return number * 5
8
-2

1
2
3
4'''

_ = Problem("challenge activity 11.1.1 part 1")

_.create("arithmetic.py", '''
def calculate(number):
    return number * 5''')

_.create("main.py", '''
import arithmetic
def calculate(number):
    return number - 5
print(arithmetic.calculate(3))
print(calculate(3))''')

_.runpy("main.py", cwd=_.folder)
_.answer('''
# Challenge Activity 11.1.1 Part 1
1) 15
2) -2
''')


"challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\narithmetic.py\nimport arithmetic\n\ndef calculate(number):\n    return number - 5\n\nprint(arithmetic.calculate(3))\nprint(calculate(3))\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\n\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\narithmetic.py\ndef calculate(number):\n    return number * 5\n8\n-2\n\n1\n2\n3\n4"

Created file: arithmetic.py with content:

def calculate(number):
    return number * 5
Created file: main.py with content:

import arithmetic
def calculate(number):
    return number - 5
print(arithmetic.calculate(3))
print(calculate(3))
Ran command: python main.py in driver_workspace/challenge_activity_11.1.1_part_1
15
-2

Saved answer to ANSWER.md:

# Challenge Activity 11.1.1 Part 1
1) 15
2) -2



In [81]:
'''challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
data.py
arithmetic.py
import arithmetic
import data

def calculate(number):
    return number * 4

print(calculate(data.medium))
print(arithmetic.calculate(data.medium))
1
2
3
4
Check
Next
1
2
3
4

Feedback?challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
data.py
arithmetic.py
small = 9
medium = 80
large = 300
1
2
3
4
Check
Next
1
2
3
4

Feedback?challenge activity
11.1.1: Enter the output of modules.
712910.5105864.qx3zqy7
Jump to level 1
Type the program's output

main.py
data.py
arithmetic.py
def calculate(number):
    return number + 2
1
2
3
4
Check
Next
1
2
3
4

Feedback?'''
_ = Problem("challenge activity 11.1.1 part 2")
_.create("data.py", '''
small = 9
medium = 80
large = 300''')
_.create("arithmetic.py", '''
def calculate(number):
    return number + 2''')
_.create("main.py", '''
import arithmetic
import data
def calculate(number):
    return number * 4
print(calculate(data.medium))
print(arithmetic.calculate(data.medium))''')
_.runpy("main.py", cwd=_.folder)
_.answer('''
# Challenge Activity 11.1.1 Part 2
1) 320
2) 82
''')

"challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\ndata.py\narithmetic.py\nimport arithmetic\nimport data\n\ndef calculate(number):\n    return number * 4\n\nprint(calculate(data.medium))\nprint(arithmetic.calculate(data.medium))\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\ndata.py\narithmetic.py\nsmall = 9\nmedium = 80\nlarge = 300\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\ndata.py\narithmetic.py\ndef calculate(number):\n    return number + 2\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?"

Created file: data.py with content:

small = 9
medium = 80
large = 300
Created file: arithmetic.py with content:

def calculate(number):
    return number + 2
Created file: main.py with content:

import arithmetic
import data
def calculate(number):
    return number * 4
print(calculate(data.medium))
print(arithmetic.calculate(data.medium))
Ran command: python main.py in driver_workspace/challenge_activity_11.1.1_part_2
320
82

Saved answer to ANSWER.md:

# Challenge Activity 11.1.1 Part 2
1) 320
2) 82



In [82]:
'challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\nimport green\nimport red\n\nprint(green.dull)\nprint(red.dull)\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\ndark = "viridian"\nbright = "forest"\nmedium = "chartreuse"\ndull = "olive"\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\ndark = "vermilion"\nbright = "crimson"\nmedium = "salmon"\ndull = "coral"\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?'
_ = Problem("challenge activity 11.1.1 part 3")
_.create("green.py", '''
dark = "viridian"
bright = "forest"
medium = "chartreuse"
dull = "olive"''')
_.create("red.py", '''
dark = "vermilion"
bright = "crimson"
medium = "salmon"
dull = "coral"''')
_.create("main.py", '''
import green
import red
print(green.dull)
print(red.dull)''')
_.runpy("main.py", cwd=_.folder)
_.answer('''
# Challenge Activity 11.1.1 Part 3
1) olive
2) coral
''')

'challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\nimport green\nimport red\n\nprint(green.dull)\nprint(red.dull)\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\ndark = "viridian"\nbright = "forest"\nmedium = "chartreuse"\ndull = "olive"\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?\nchallenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program\'s output\n\nmain.py\ngreen.py\nred.py\ndark = "vermilion"\nbright = "crimson"\nmedium = "salmon"\ndull = "coral"\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?'

Created file: green.py with content:

dark = "viridian"
bright = "forest"
medium = "chartreuse"
dull = "olive"
Created file: red.py with content:

dark = "vermilion"
bright = "crimson"
medium = "salmon"
dull = "coral"
Created file: main.py with content:

import green
import red
print(green.dull)
print(red.dull)
Ran command: python main.py in driver_workspace/challenge_activity_11.1.1_part_3
olive
coral

Saved answer to ANSWER.md:

# Challenge Activity 11.1.1 Part 3
1) olive
2) coral



In [83]:
"challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\nimport first\n\ndef fct_a(number):\n    return number ** 2\n\ndef fct_b(number):\n    return number * 1\n\ndef fct_c(number):\n    return fct_a(number) - fct_b(number)\n\nprint(fct_c(7))\nprint(first.fct_c(7))\nprint(first.fct_d(7))\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\nimport second\n\ndef fct_a(number):\n    return number + 9\n\ndef fct_b(number):\n    return number * 6\n\ndef fct_c(number):\n    return fct_a(number) - fct_b(number)\n\ndef fct_d(number):\n    return second.fct_c(number)\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\ndef fct_a(number):\n    return number - 3\n\ndef fct_b(number):\n    return number + 8\n\ndef fct_c(number):\n    return number * 2\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?"
_ = Problem("challenge activity 11.1.1 part 4")
_.create("first.py", '''
import second

def fct_a(number):
    return number + 3

def fct_b(number):
    return number * 8

def fct_c(number):
    return fct_a(number) - fct_b(number)

def fct_d(number):
    return second.fct_c(number)''')
_.create("second.py", '''
def fct_a(number):
    return number - 6

def fct_b(number):
    return number + 1

def fct_c(number):
    return number * 7''')
_.create("main.py", '''
import first

def fct_a(number):
    return number ** 2

def fct_b(number):
    return number * 5

def fct_c(number):
    return fct_a(number) - fct_b(number)

print(fct_c(4))
print(first.fct_c(4))
print(first.fct_d(4))''')
_.runpy("main.py", cwd=_.folder)
_.answer('''
# Challenge Activity 11.1.1 Part 4
1) 0
2) -25
3) -28
''')

"challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\nimport first\n\ndef fct_a(number):\n    return number ** 2\n\ndef fct_b(number):\n    return number * 1\n\ndef fct_c(number):\n    return fct_a(number) - fct_b(number)\n\nprint(fct_c(7))\nprint(first.fct_c(7))\nprint(first.fct_d(7))\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nmain.py\nfirst.py\nsecond.py\nimport second\n\ndef fct_a(number):\n    return number + 9\n\ndef fct_b(number):\n    return number * 6\n\ndef fct_c(number):\n    return fct_a(number) - fct_b(number)\n\ndef fct_d(number):\n    return second.fct_c(number)\n1\n2\n3\n4\nCheck\nNext\n1\n2\n3\n4\n\nFeedback?challenge activity\n11.1.1: Enter the output of modules.\n712910.5105864.qx3zqy7\nJump to level 1\nType the program's output\n\nm

Created file: first.py with content:

import second

def fct_a(number):
    return number + 3

def fct_b(number):
    return number * 8

def fct_c(number):
    return fct_a(number) - fct_b(number)

def fct_d(number):
    return second.fct_c(number)
Created file: second.py with content:

def fct_a(number):
    return number - 6

def fct_b(number):
    return number + 1

def fct_c(number):
    return number * 7
Created file: main.py with content:

import first

def fct_a(number):
    return number ** 2

def fct_b(number):
    return number * 5

def fct_c(number):
    return fct_a(number) - fct_b(number)

print(fct_c(4))
print(first.fct_c(4))
print(first.fct_d(4))
Ran command: python main.py in driver_workspace/challenge_activity_11.1.1_part_4
-4
-25
28

Saved answer to ANSWER.md:

# Challenge Activity 11.1.1 Part 4
1) 0
2) -25
3) -28



In [91]:
'''challenge activity
11.1.2: Modules.
712910.5105864.qx3zqy7

Jump to level 1
Integers base_in_cm and height_in_cm are read from input, representing a triangle's base and height in centimeters. Module si_units defines cm_to_mm() and module tri_shape defines area(). Import modules si_units and tri_shape into main.py.

Click here for example
Ex: If the input is:
8
3
then the output is:

8 cm is 80 mm
3 cm is 30 mm
Triangle area in mm^2: 1200.0
'''
_q = Problem("challenge activity 11.1.2 part 1")
_q.create("si_units.py", '''
def cm_to_mm(cm):
    return cm * 10''')
_q.create("tri_shape.py", '''
def area(base, height):
    return base * height / 2''')
_q.create("main.py", '''
import si_units
import tri_shape
base_in_cm = int(input())
height_in_cm = int(input())

base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)

print(f"{base_in_cm} cm is {base_in_mm} mm")
print(f"{height_in_cm} cm is {height_in_mm} mm")
print(f"Triangle area in mm^2: {tri_shape.area(base_in_mm, height_in_mm)}")''')

_q.runpy("main.py", inputs=["8", "3"])
_q.answer('''
# Challenge Activity 11.1.2 Part 1
1) import si_units
import tri_shape
2) 8 cm is 80 mm
3) 3 cm is 30 mm
4) Triangle area in mm^2: 1200.0
''')


"challenge activity\n11.1.2: Modules.\n712910.5105864.qx3zqy7\n\nJump to level 1\nIntegers base_in_cm and height_in_cm are read from input, representing a triangle's base and height in centimeters. Module si_units defines cm_to_mm() and module tri_shape defines area(). Import modules si_units and tri_shape into main.py.\n\nClick here for example\nEx: If the input is:\n8\n3\nthen the output is:\n\n8 cm is 80 mm\n3 cm is 30 mm\nTriangle area in mm^2: 1200.0\n"

Created file: si_units.py with content:

def cm_to_mm(cm):
    return cm * 10
Created file: tri_shape.py with content:

def area(base, height):
    return base * height / 2
Created file: main.py with content:

import si_units
import tri_shape
base_in_cm = int(input())
height_in_cm = int(input())

base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)

print(f"{base_in_cm} cm is {base_in_mm} mm")
print(f"{height_in_cm} cm is {height_in_mm} mm")
print(f"Triangle area in mm^2: {tri_shape.area(base_in_mm, height_in_mm)}")
Ran command: python main.py in driver_workspace/challenge_activity_11.1.2_part_1
8 cm is 80 mm
3 cm is 30 mm
Triangle area in mm^2: 1200.0

Saved answer to ANSWER.md:

# Challenge Activity 11.1.2 Part 1
1) import si_units
import tri_shape
2) 8 cm is 80 mm
3) 3 cm is 30 mm
4) Triangle area in mm^2: 1200.0



In [95]:
'''challenge activity
11.1.2: Modules.
712910.5105864.qx3zqy7

Jump to level 1
Integers base_in_cm and height_in_cm are read from input, representing a triangle's base and height in centimeters. si_units's cm_to_mm() converts a measurement from centimeters to millimeters.

Assign area_cm2 with the value returned by tri_shape's area() called with base_in_cm and height_in_cm as the arguments, respectively.
Find the triangle's base and height in millimeters. Then, assign area_mm2 with the value returned by tri_shape's area() called with the triangle's base and height in millimeters as the arguments, respectively.
Click here for example
Ex: If the input is:
7
8
then the output is:

Triangle area is 28.0 cm^2 or 2800.0 mm^2.


main.py
import si_units
import tri_shape

base_in_cm = int(input())
height_in_cm = int(input())

""" Your code goes here """

print(f"Triangle area is {area_cm2} cm^2 or {area_mm2} mm^2.")
si_units.py
def cm_to_mm(cm):
	return cm * 10

tri_shape.py
def area(base, height):
	return base * height / 2
'''
_ = Problem("challenge activity 11.1.2 part 2")
_.create("si_units.py", '''
def cm_to_mm(cm):
    return cm * 10''')
_.create("tri_shape.py", '''
def area(base, height):
    return base * height / 2''')
_.create("main.py", '''
import si_units
import tri_shape

base_in_cm = int(input())
height_in_cm = int(input())

area_cm2 = tri_shape.area(base_in_cm, height_in_cm)

base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)

area_mm2 = tri_shape.area(base_in_mm, height_in_mm)

print(f"Triangle area is {area_cm2} cm^2 or {area_mm2} mm^2.")''')
_.runpy("main.py", inputs=["7", "8"])
_.answer('''
# Challenge Activity 11.1.2 Part 2
1) area_cm2 = tri_shape.area(base_in_cm, height_in_cm)
2) base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)
area_mm2 = tri_shape.area(base_in_mm, height_in_mm)
3) Triangle area is 28.0 cm^2 or 2800.0 mm^2.
''')

'challenge activity\n11.1.2: Modules.\n712910.5105864.qx3zqy7\n\nJump to level 1\nIntegers base_in_cm and height_in_cm are read from input, representing a triangle\'s base and height in centimeters. si_units\'s cm_to_mm() converts a measurement from centimeters to millimeters.\n\nAssign area_cm2 with the value returned by tri_shape\'s area() called with base_in_cm and height_in_cm as the arguments, respectively.\nFind the triangle\'s base and height in millimeters. Then, assign area_mm2 with the value returned by tri_shape\'s area() called with the triangle\'s base and height in millimeters as the arguments, respectively.\nClick here for example\nEx: If the input is:\n7\n8\nthen the output is:\n\nTriangle area is 28.0 cm^2 or 2800.0 mm^2.\n\n\nmain.py\nimport si_units\nimport tri_shape\n\nbase_in_cm = int(input())\nheight_in_cm = int(input())\n\n""" Your code goes here """\n\nprint(f"Triangle area is {area_cm2} cm^2 or {area_mm2} mm^2.")\nsi_units.py\ndef cm_to_mm(cm):\n\treturn cm * 1

Created file: si_units.py with content:

def cm_to_mm(cm):
    return cm * 10
Created file: tri_shape.py with content:

def area(base, height):
    return base * height / 2
Created file: main.py with content:

import si_units
import tri_shape

base_in_cm = int(input())
height_in_cm = int(input())

area_cm2 = tri_shape.area(base_in_cm, height_in_cm)

base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)

area_mm2 = tri_shape.area(base_in_mm, height_in_mm)

print(f"Triangle area is {area_cm2} cm^2 or {area_mm2} mm^2.")
Ran command: python main.py in driver_workspace/challenge_activity_11.1.2_part_2
Triangle area is 28.0 cm^2 or 2800.0 mm^2.

Saved answer to ANSWER.md:

# Challenge Activity 11.1.2 Part 2
1) area_cm2 = tri_shape.area(base_in_cm, height_in_cm)
2) base_in_mm = si_units.cm_to_mm(base_in_cm)
height_in_mm = si_units.cm_to_mm(height_in_cm)
area_mm2 = tri_shape.area(base_in_mm, height_in_mm)
3) Triangle area is 28.0 cm^2 or 2800.0 mm^2.



In [98]:
'''challenge activity
11.1.2: Modules.
712910.5105864.qx3zqy7

Jump to level 1
Perform the following tasks:

Assign has_color with the value returned by search's find() called with one_sentence and colors's search_list as the arguments, respectively.
Assign has_person_and_clothing with the value returned by combining the following, using logical AND:
search's find() called with one_sentence and persons's search_list as the arguments, respectively.
search's find() called with one_sentence and clothing's search_list as the arguments, respectively.
Click here for example
Ex: If the input is A pink t-shirt was sent to Del., then the output is:

The sentence mentions a color.
The sentence mentions a person and a piece of clothing.


main.py
import persons
import colors
import clothing
import search

one_sentence = input()

""" Your code goes here """

if has_color:
	print("The sentence mentions a color.")

if has_person_and_clothing:
	print("The sentence mentions a person and a piece of clothing.")
persons.py
search_list = ["Del", "Noa", "Ada"]
colors.py
search_list = ["red", "white", "pink"]
clothing.py
search_list = ["shirt", "jacket", "t-shirt"]
search.py
def find(one_sentence, search_list):
    words = one_sentence[:-1].split()
    for word in words:
        if word in search_list:
            return True
    return False
1

2

3

Check

Next level
1
2
3

Feedback?'''
_ = Problem("challenge activity 11.1.2 part 3")
_.create("persons.py", '''
search_list = ["Del", "Noa", "Ada"]''')
_.create("colors.py", '''
search_list = ["red", "white", "pink"]''')
_.create("clothing.py", '''
search_list = ["shirt", "jacket", "t-shirt"]''')
_.create("search.py", '''
def find(one_sentence, search_list):
    words = one_sentence[:-1].split()
    for word in words:
        if word in search_list:
            return True
    return False''')
_.create("main.py", '''
import persons
import colors
import clothing
import search
one_sentence = input()
has_color = search.find(one_sentence, colors.search_list)
has_person_and_clothing = search.find(one_sentence, persons.search_list) and search.find(one_sentence, clothing.search_list)
if has_color:
    print("The sentence mentions a color.") 
if has_person_and_clothing:
    print("The sentence mentions a person and a piece of clothing.")''')
_.runpy("main.py", inputs=["A pink t-shirt was sent to Del."])
_.answer('''
# Challenge Activity 11.1.2 Part 3
1) has_color = search.find(one_sentence, colors.search_list)
2) has_person_and_clothing = search.find(one_sentence, persons.search_list) and search.find(one_sentence, clothing.search_list)
3) The sentence mentions a color.
The sentence mentions a person and a piece of clothing.
''')

'challenge activity\n11.1.2: Modules.\n712910.5105864.qx3zqy7\n\nJump to level 1\nPerform the following tasks:\n\nAssign has_color with the value returned by search\'s find() called with one_sentence and colors\'s search_list as the arguments, respectively.\nAssign has_person_and_clothing with the value returned by combining the following, using logical AND:\nsearch\'s find() called with one_sentence and persons\'s search_list as the arguments, respectively.\nsearch\'s find() called with one_sentence and clothing\'s search_list as the arguments, respectively.\nClick here for example\nEx: If the input is A pink t-shirt was sent to Del., then the output is:\n\nThe sentence mentions a color.\nThe sentence mentions a person and a piece of clothing.\n\n\nmain.py\nimport persons\nimport colors\nimport clothing\nimport search\n\none_sentence = input()\n\n""" Your code goes here """\n\nif has_color:\n\tprint("The sentence mentions a color.")\n\nif has_person_and_clothing:\n\tprint("The sentenc

Created file: persons.py with content:

search_list = ["Del", "Noa", "Ada"]
Created file: colors.py with content:

search_list = ["red", "white", "pink"]
Created file: clothing.py with content:

search_list = ["shirt", "jacket", "t-shirt"]
Created file: search.py with content:

def find(one_sentence, search_list):
    words = one_sentence[:-1].split()
    for word in words:
        if word in search_list:
            return True
    return False
Created file: main.py with content:

import persons
import colors
import clothing
import search
one_sentence = input()
has_color = search.find(one_sentence, colors.search_list)
has_person_and_clothing = search.find(one_sentence, persons.search_list) and search.find(one_sentence, clothing.search_list)
if has_color:
    print("The sentence mentions a color.") 
if has_person_and_clothing:
    print("The sentence mentions a person and a piece of clothing.")
Ran command: python main.py in driver_workspace/challenge_activity_11.1.2_part_3
The sentenc

In [106]:
class scratchwork:
    _=Problem("scratchwork")

    _.create("test.py", '''def fct():
        return "Hello from test.py!"''')
    _.runpy("test.py", cwd=_.folder)

    print(_.get_file("test.py"))

Created file: test.py with content:
def fct():
        return "Hello from test.py!"
Ran command: python test.py in driver_workspace/scratchwork

def fct():
        return "Hello from test.py!"


11.2 Finding modules

https://learn.zybooks.com/zybook/CPPCS2520NguyenSpring2026/chapter/11/section/2

In [109]:
'''11.2 Finding modules
Importing a module begins a search to find the corresponding file on the computer's file system. The interpreter first checks for a matching built-in module. A built-in module comes pre-installed with Python; examples of built-in modules include sys, time, and math. If no matching built-in module is found, then the interpreter searches for the necessary module in sys.path, a built-in Python variable located in the sys module that has a list of directories containing modules. A programmer must be careful to not give a name to a module that is already used by a built-in module. In such cases, the interpreter would load the built-in module because built-in names are checked first.

The sys.path variable initially contains the following directories:


The directory of the executing script.
A list of directories specified by the environment variable PYTHONPATH.
The directory where Python is installed.
For simple programs, a module might be placed in the same directory. Larger projects might contain tens or hundreds of modules or use third-party modules located in different directories. In such cases, a programmer might set the environment variable PYTHONPATH in the operating system. An operating system environment variable is much like a variable in a Python script, except that an environment variable is stored by the computer's operating system and can be accessed by every program running on the computer. In Windows, a user can set the value of PYTHONPATH permanently through the control panel, or temporarily on a single instance of a command terminal (cmd.exe) using the command set PYTHONPATH="c:\dir1;c:\other\directory".

activity
11.2.1: Finding modules.
1)
When an import statement executes, the interpreter immediately checks the current directory for a matching file.
2)
The environment variable PYTHONPATH can be set to specify optional directories where modules are located.
3)
math.py is a good name for a new module.

Feedback?
How was this section?

|


Provide section feedback'''
_ = Problem("activity 11.2.1")
_.create("script.py", '''
import math
print(math.sqrt(16))
''')
_.runpy("script.py", cwd=_.folder)
_.answer('''
# Activity 11.2.1
1) False
2) True
3) False
''')


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_69779/1390622080.py:1: SyntaxWarning: invalid escape sequence '\d'
  '''11.2 Finding modules


'11.2 Finding modules\nImporting a module begins a search to find the corresponding file on the computer\'s file system. The interpreter first checks for a matching built-in module. A built-in module comes pre-installed with Python; examples of built-in modules include sys, time, and math. If no matching built-in module is found, then the interpreter searches for the necessary module in sys.path, a built-in Python variable located in the sys module that has a list of directories containing modules. A programmer must be careful to not give a name to a module that is already used by a built-in module. In such cases, the interpreter would load the built-in module because built-in names are checked first.\n\nThe sys.path variable initially contains the following directories:\n\n\nThe directory of the executing script.\nA list of directories specified by the environment variable PYTHONPATH.\nThe directory where Python is installed.\nFor simple programs, a module might be placed in the same 

Created file: script.py with content:

import math
print(math.sqrt(16))

Ran command: python script.py in driver_workspace/activity_11.2.1
4.0

Saved answer to ANSWER.md:

# Activity 11.2.1
1) False
2) True
3) False



In [ ]:
_= Problem("scratchwork")
